In [1]:
# 1. 必要なライブラリのインストール
!pip install ripser persim umap-learn scipy scikit-learn

import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import pairwise_distances
import umap
from ripser import ripser
from persim import plot_diagrams
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# 2. 実験設定とディレクトリ作成
OUT_DIR = "H6_experiment_results"
os.makedirs(OUT_DIR, exist_ok=True)

# 試行回数（総生成要素数）の6分割スケール
trial_scales = [1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

N_PER_GROUP = 100
N_SAMPLES = N_PER_GROUP * 2 # Group A, B 合計200
results = []

print("=== H6計算実験（TDAによるキメラクラスターの事後証明）を開始します ===")

def clr_transform(X, epsilon=1e-10):
    X_eps = X + epsilon
    log_X = np.log(X_eps)
    return log_X - np.mean(log_X, axis=1, keepdims=True)

# 3. メインループ
for i, total_trials in enumerate(trial_scales):
    step = i + 1
    D_features = max(10, total_trials // N_SAMPLES)
    print(f"\n[Step {step}/6] 試行回数: {total_trials:,} (N={N_SAMPLES}, D={D_features})")

    np.random.seed(42 + step)

    # --- A. 真の異質集団（2つの異なる確率分布）の生成 ---
    # Group A: 低い値に分布, Group B: 高い値に分布（意味的に離れた集団を模倣）
    X_A = np.random.uniform(0.01, 0.2, size=(N_PER_GROUP, D_features))
    X_B = np.random.uniform(0.7, 0.9, size=(N_PER_GROUP, D_features))
    X_raw = np.vstack([X_A, X_B])

    # --- B. ハブ（過分散）の注入による空間破壊の準備 ---
    X_skewed = X_raw.copy()
    hub_dims = max(1, int(D_features * 0.05))
    skew_multiplier = 10.0 + (step * 5.0)
    # 両方のグループに共通の高頻度ノイズ（ハブ）を注入
    X_skewed[:, :hub_dims] *= skew_multiplier

    # --- C. 組成データ（確率単体）への正規化 ---
    X_simplex = X_skewed / np.sum(X_skewed, axis=1, keepdims=True)

    # --- D. 定石パイプライン（コサイン + UMAP）の実行（キメラの生成） ---
    dist_cos = pairwise_distances(X_simplex + 1e-10, metric='cosine')
    reducer = umap.UMAP(n_components=2, metric='precomputed', random_state=42)
    umap_emb = reducer.fit_transform(dist_cos)

    # UMAP上での Group A と Group B の重心間距離を計算（融合度の測定）
    centroid_A_umap = np.mean(umap_emb[:N_PER_GROUP], axis=0)
    centroid_B_umap = np.mean(umap_emb[N_PER_GROUP:], axis=0)
    umap_cluster_distance = np.linalg.norm(centroid_A_umap - centroid_B_umap)

    # --- E. TDAによる事後監査（Sanity Check） ---
    # 高次元空間における正しい計量（Aitchison距離）でPHを計算
    dist_aitchison = pairwise_distances(clr_transform(X_simplex), metric='euclidean')
    ph_res = ripser(dist_aitchison, distance_matrix=True, maxdim=1)['dgms']

    # H0（連結成分）の寿命（death）を取得
    h0_deaths = ph_res[0][:, 1]
    # 無限大（全体の連結成分）を除外するため、有限な寿命をソート
    finite_h0_deaths = np.sort(h0_deaths[h0_deaths < np.inf])

    # キメラ・ギャップ（2番目に長いH0の寿命）：
    # 単一同質集団ならノイズなのですぐ死ぬ（値が小さい）。
    # 異質集団の融合（キメラ）なら、2つの塊が繋がるまでの巨大な距離が残る。
    chimera_gap = finite_h0_deaths[-1] if len(finite_h0_deaths) > 0 else 0.0

    # 結果の記録
    results.append({
        'Step': step,
        'Dimensions': D_features,
        'UMAP_Visual_Distance': umap_cluster_distance,
        'TDA_Chimera_Gap': chimera_gap
    })

    # --- F. グラフ出力 ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Step {step}: Detecting the Chimera Cluster (D={D_features})', fontsize=16)

    # 1. UMAP散布図（キメラの視覚的確認）
    axes[0].scatter(umap_emb[:N_PER_GROUP, 0], umap_emb[:N_PER_GROUP, 1], label='Group A', alpha=0.7)
    axes[0].scatter(umap_emb[N_PER_GROUP:, 0], umap_emb[N_PER_GROUP:, 1], label='Group B', alpha=0.7, color='red')
    axes[0].set_title(f'Illusion (UMAP with Cosine)\nCentroid Distance: {umap_cluster_distance:.3f}')
    axes[0].legend()

    # 2. パーシステンス図（TDAによる真実の暴露）
    plot_diagrams(ph_res, ax=axes[1])
    axes[1].set_title(f'Sanity Check (TDA on High-D)\nChimera Gap (H0): {chimera_gap:.3f}')
    # キメラギャップ（一番上の赤い点）を強調
    axes[1].axhline(y=chimera_gap, color='gray', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/trial_step_{step}.png")
    plt.close()

# 4. 集計とまとめグラフ
df_results = pd.DataFrame(results)
df_results.to_csv(f"{OUT_DIR}/summary_results.csv", index=False)

fig, ax1 = plt.subplots(figsize=(9, 6))
ax1.set_xlabel('Total Trials (log scale)')
ax1.set_xscale('log')

# UMAP上での距離（視覚的な融合度）
ax1.set_ylabel('Visual Distance in UMAP (Lower = Merged into Chimera)', color='blue')
ax1.plot(trial_scales, df_results['UMAP_Visual_Distance'], marker='o', color='blue', label='UMAP Apparent Distance')
ax1.tick_params(axis='y', labelcolor='blue')

# TDAによるキメラギャップ（数学的な分離度）
ax2 = ax1.twinx()
ax2.set_ylabel('TDA Chimera Gap (Higher = Proof of Heterogeneity)', color='red')
ax2.plot(trial_scales, df_results['TDA_Chimera_Gap'], marker='s', color='red', label='High-D TDA Chimera Gap')
ax2.tick_params(axis='y', labelcolor='red')

fig.suptitle('H6: Topological Sanity Check Rejects UMAP Chimeras', fontsize=14)
plt.title('While UMAP visually merges groups (blue line drops), TDA mathematically proves they are distinct (red line rises)', fontsize=10)
fig.tight_layout()
plt.grid(True, which="both", ls="--", alpha=0.5)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')

plt.savefig(f"{OUT_DIR}/summary_h6_plot.png")
plt.close()

# 5. ZIP圧縮とダウンロード
shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print(f"\n実験完了。全データを {OUT_DIR}.zip に圧縮しました。ダウンロードを開始します...")
files.download(f"{OUT_DIR}.zip")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.1/842.1 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.7 MB/s eta 0:00:00
  Created wheel for hopcroftkarp: filename=hopcroftkarp-1.2.5-py2.py3-none-any.whl size=18187 sha256=b3b1b691139068b9f3cccc34911a2baa1457f72a8c4f581f887346b61ca770a4
  Stored in directory: /root/.cache/pip/wheels/b3/32/77/3c78fadb9f917d5a5546ed0409574012401a83c544ec95999a
Successfully built hopcroftkarp
=== H6計算実験（TDAによるキメラクラスターの事後証明）を開始します ===

[Step 1/6] 試行回数: 1,000 (N=200, D=10)

[Step 2/6] 試行回数: 10,000 (N=200, D=50)

[Step 3/6] 試行回数: 100,000 (N=200, D=500)

[Step 4/6] 試行回数: 1,000,000 (N=200, D=5000)

[Step 5/6] 試行回数: 5,000,000 (N=200, D=25000)

[Step 6/6] 試行回数: 10,000,000 (N=200, D=50000)

実験完了。全データを H6_experiment_results.zip に圧縮しました。ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>